In [1]:
import pandas as pd
import pandas_gbq
from google.cloud import bigquery
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from tqdm.auto import tqdm
from google.oauth2 import service_account
import numpy as np

# 1. 신분증(JSON 키) 경로 지정
KEY_PATH = '../google_key.json'

# 2. 인증 객체 생성
credentials = service_account.Credentials.from_service_account_file(KEY_PATH)

# 3. 프로젝트 ID 설정
project_id = 'gdelt-analysis-494301'

# 4. 데이터 불러오기
query = "SELECT SQLDATE FROM `gdelt-bq.full.events` LIMIT 5"
df = pandas_gbq.read_gbq(query, project_id=project_id, credentials=credentials)

print("인증 성공! 데이터를 가져왔습니다.")

Downloading: 100%|██████████|
인증 성공! 데이터를 가져왔습니다.


In [2]:
# 1. 고위험군 CAMEO 코드 리스트
target_cameo_codes = [
    '150', '151', '152', '153', '154', '155', 
    '190', '191', '192', '193', '194', '195', '196', 
    '200', '201', '202', '203', '204'
]
formatted_codes = ", ".join([f"'{code}'" for code in target_cameo_codes])

query = f"""
SELECT 
    SQLDATE, 
    EventCode,
    GoldsteinScale, 
    NumMentions, 
    AvgTone,
    ActionGeo_Type,
    ActionGeo_Lat, 
    ActionGeo_Long, 
    SOURCEURL
FROM `gdelt-bq.full.events`
WHERE SQLDATE >= 20130401 
  AND (
    (Actor1CountryCode = 'CHN' AND Actor2CountryCode = 'TWN') OR 
    (Actor1CountryCode = 'TWN' AND Actor2CountryCode = 'CHN')
  )
  AND EventCode IN ({formatted_codes})
  AND IsRootEvent = 1                 -- 핵심 사건만 필터링
  AND ActionGeo_Type IN (3, 4, 5)
"""

# 2. 데이터 저장
df_gdelt1 = pandas_gbq.read_gbq(query, project_id=project_id, credentials=credentials)

print("데이터 불러오기 완료")
display(df_gdelt1.head())


Downloading: 100%|██████████|
데이터 불러오기 완료


,SQLDATE,EventCode,GoldsteinScale,NumMentions,AvgTone,ActionGeo_Type,ActionGeo_Lat,ActionGeo_Long,SOURCEURL
0,20260510,194,-10.0,16,-2.429197,4,24.9139,118.586,https://www.taipeitimes.com/News/front/archive...
1,20260510,154,-7.2,1,-3.183521,4,25.0478,121.532,https://news.ltn.com.tw/news/focus/breakingnew...
2,20260510,154,-7.2,1,-3.183521,4,25.0478,121.532,https://news.ltn.com.tw/news/focus/breakingnew...
3,20260514,190,-10.0,10,-2.950820,4,39.9289,116.388,https://www.armytimes.com/news/pentagon-congre...
4,20260514,193,-10.0,10,-2.950820,4,39.9289,116.388,https://www.armytimes.com/news/pentagon-congre...


In [4]:
df_gdelt1.to_csv("raw/gdelt_raw1.csv", index=False)
print(f"저장 완료: {len(df_gdelt1)}행")

저장 완료: 11941행


In [5]:
target_cameo_codes = [
    '150', '151', '152', '153', '154', '155', 
    '190', '191', '192', '193', '194', '195', '196', 
    '200', '201', '202', '203', '204'
]
formatted_codes = ", ".join([f"'{code}'" for code in target_cameo_codes])

query = f"""
SELECT 
    SQLDATE,
    EventCode,
    QuadClass,
    GoldsteinScale,
    NumMentions,
    NumArticles,
    AvgTone,
    Actor1CountryCode,
    Actor2CountryCode,
    Actor1Type1Code,
    Actor2Type1Code,
    Actor1Geo_Type,
    Actor1Geo_Lat,
    Actor1Geo_Long,
    Actor2Geo_Type,
    Actor2Geo_Lat,
    Actor2Geo_Long,
    ActionGeo_Type,
    ActionGeo_Lat,
    ActionGeo_Long,
    SOURCEURL
FROM `gdelt-bq.full.events`
WHERE SQLDATE >= 20130401 
  AND (
    (Actor1CountryCode = 'CHN' AND Actor2CountryCode = 'TWN') OR 
    (Actor1CountryCode = 'TWN' AND Actor2CountryCode = 'CHN')
  )
  AND EventCode IN ({formatted_codes})
  AND IsRootEvent = 1
  AND ActionGeo_Type IN (3, 4, 5)
"""

# 2. 데이터 불러오기
df_gdelt2 = pandas_gbq.read_gbq(query, project_id=project_id, credentials=credentials)

print("데이터 불러오기 완료")
display(df_gdelt2.head())

Downloading: 100%|██████████|
데이터 불러오기 완료


,SQLDATE,EventCode,QuadClass,GoldsteinScale,NumMentions,NumArticles,AvgTone,Actor1CountryCode,Actor2CountryCode,Actor1Type1Code,...,Actor1Geo_Type,Actor1Geo_Lat,Actor1Geo_Long,Actor2Geo_Type,Actor2Geo_Lat,Actor2Geo_Long,ActionGeo_Type,ActionGeo_Lat,ActionGeo_Long,SOURCEURL
0,20160522,150,4,-7.2,5,5,-0.806452,CHN,TWN,None,...,4,39.9289,116.388,4,25.0327,121.275,4,39.9289,116.388,http://bilbaoya.com/2016/05/23/taiwan-swears-i...
1,20160522,150,4,-7.2,5,5,0.634249,CHN,TWN,None,...,4,25.0327,121.275,4,25.0327,121.275,4,25.0327,121.275,http://fait-religieux.com/2016/05/22/taiwan-sw...
2,20260512,190,4,-10.0,25,15,-1.751929,TWN,CHN,None,...,4,24.0000,119.000,4,39.9289,116.388,4,39.9289,116.388,https://www.yahoo.com/news/articles/key-events...
3,20260512,194,4,-10.0,14,11,-1.751929,TWN,CHN,None,...,4,24.0000,119.000,4,39.9289,116.388,4,39.9289,116.388,https://www.yahoo.com/news/articles/key-events...
4,20260512,192,4,-9.5,4,4,-1.831123,CHN,TWN,None,...,4,25.0478,121.532,4,39.9289,116.388,4,39.9289,116.388,https://www.hindustantimes.com/world-news/ahea...


In [6]:
df_gdelt2.to_csv("raw/gdelt_raw2.csv", index=False)
print(f"저장 완료: {len(df_gdelt2)}행")

저장 완료: 11941행


In [ ]:
dedup_v1 = df_gdelt1.drop_duplicates(subset=[
    'SQLDATE', 'EventCode', 'ActionGeo_Lat', 'ActionGeo_Long'
])

dedup_v2 = df_gdelt2.drop_duplicates(subset=[
    'SQLDATE', 'EventCode', 'ActionGeo_Lat', 'ActionGeo_Long',
    'Actor2Geo_Lat', 'Actor2Geo_Long'
])

print(f"원본:                     {len(df_gdelt1)}")
print(f"v1 기준 (4컬럼):          {len(dedup_v1)}")
print(f"v2 기준 (Actor2Geo 추가): {len(dedup_v2)}")

원본:                     11941
v1 기준 (4컬럼):          6890
v2 기준 (Actor2Geo 추가): 8840


In [9]:
dedup_v1['SQLDATE'] = pd.to_datetime(dedup_v1['SQLDATE'].astype(str), format='%Y%m%d')

C:\Users\Jua\AppData\Local\Temp\ipykernel_4468\1653496930.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dedup_v1['SQLDATE'] = pd.to_datetime(dedup_v1['SQLDATE'].astype(str), format='%Y%m%d')


In [11]:
# df_gdelt2 기준으로 확인
print(f"df_gdelt1 행 수: {len(df_gdelt1)}")
print(f"df_gdelt2 행 수: {len(df_gdelt2)}")

# TWN→CHN 이벤트 Actor2Geo 분포
twn_to_chn = df_gdelt2[df_gdelt2['Actor1CountryCode'] == 'TWN'].copy()
print(f"\nTWN→CHN 이벤트 수: {len(twn_to_chn)}")

unique_actor2 = twn_to_chn[['Actor2Geo_Lat', 'Actor2Geo_Long']].drop_duplicates()
print(f"고유 Actor2Geo 좌표 수: {len(unique_actor2)}")

coastal = unique_actor2[
    unique_actor2['Actor2Geo_Lat'].between(20, 28) &
    unique_actor2['Actor2Geo_Long'].between(116, 122)
]
print(f"연안 범위 내: {len(coastal)}건")
print(f"연안 범위 밖: {len(unique_actor2) - len(coastal)}건")

df_gdelt1 행 수: 11941
df_gdelt2 행 수: 11941

TWN→CHN 이벤트 수: 4599
고유 Actor2Geo 좌표 수: 368
연안 범위 내: 99건
연안 범위 밖: 269건


In [12]:
outside_coastal = unique_actor2[
    ~(
        unique_actor2['Actor2Geo_Lat'].between(20, 28) &
        unique_actor2['Actor2Geo_Long'].between(116, 122)
    )
]

# 역지오코딩 전에 좌표 분포 먼저 시각적으로 확인
print("범위 밖 좌표 샘플:")
print(outside_coastal.head(20).to_string())

# 대륙별 분류
print("\n위도/경도 범위별 분류:")
print(f"대만 범위 (21~26, 118~122): {len(outside_coastal[outside_coastal['Actor2Geo_Lat'].between(21,26) & outside_coastal['Actor2Geo_Long'].between(118,122)])}개")
print(f"중국 내륙 (28~45, 100~135): {len(outside_coastal[outside_coastal['Actor2Geo_Lat'].between(28,45) & outside_coastal['Actor2Geo_Long'].between(100,135)])}개")
print(f"기타: {len(outside_coastal[~(outside_coastal['Actor2Geo_Lat'].between(18,45) & outside_coastal['Actor2Geo_Long'].between(100,135))])}개")

범위 밖 좌표 샘플:
     Actor2Geo_Lat  Actor2Geo_Long
2          39.9289      116.388000
41         41.0000      123.000000
43         48.8667        2.333330
45         15.0000      115.000000
50         21.4667       87.016700
83         32.5630     -106.570000
95         24.4533      114.935000
96         23.1167      113.250000
100        38.8951      -77.036400
106        32.9447      111.031000
117        24.1098      113.005000
139        39.9044      116.391000
154        43.0999      129.768000
166        51.5000       -0.116667
170        35.0000      105.000000
171        40.7703      123.792000
257        45.6167      -61.966700
271        31.2222      121.458000
286        29.5792      116.227000
304        22.4285      112.857000

위도/경도 범위별 분류:
대만 범위 (21~26, 118~122): 0개
중국 내륙 (28~45, 100~135): 114개
기타: 96개


In [13]:
# TWN→CHN ActionGeo 연안 범위 확인
twn_action = twn_to_chn[['ActionGeo_Lat', 'ActionGeo_Long']].drop_duplicates()

coastal_action = twn_action[
    twn_action['ActionGeo_Lat'].between(20, 28) &
    twn_action['ActionGeo_Long'].between(116, 122)
]

print(f"TWN→CHN ActionGeo 고유 좌표: {len(twn_action)}개")
print(f"연안 범위 내: {len(coastal_action)}개 ({len(coastal_action)/len(twn_action)*100:.1f}%)")

TWN→CHN ActionGeo 고유 좌표: 358개
연안 범위 내: 118개 (33.0%)
